# Exploración individual de datos | OT Consulting
**Nombre:** ____________________  
**Corte del respaldo:** 17 de noviembre de 2026

Coloca el respaldo autorizado junto al notebook o en `upload/` y ejecuta todas las celdas con Python 3. Solo se usa la biblioteca estándar. Las salidas contienen perfiles y conteos agregados. Consulta [el informe](Informe_OTC.md) para las reglas, el diccionario y las preguntas.
## 1. Inventario de fuentes y decisión de lectura
El notebook recibido es una referencia metodológica. Declara diez archivos Excel bajo `/content/` y genera análisis de personal, nómina, proyectos, activos y habilidades. Los Excel no se entregaron. Este cuaderno ejecuta el respaldo JSON adjunto. La nómina se observa en colaboradores y los proyectos y las entregas de equipo vienen anidados. El JSON también contiene cuentas de aplicación y un contador operativo. Se revisa el esquema de cuentas y se omiten los valores del campo de autenticación.

In [1]:
from collections import Counter
from datetime import date
from hashlib import sha256
from pathlib import Path
import json
import sqlite3

nombre = 'otc_backup_2026-11-17.json'
rutas = [Path(nombre), Path('upload') / nombre, Path('..') / nombre,
         Path('..') / 'upload' / nombre]
ruta = next((p for p in rutas if p.is_file()), None)
if ruta is None:
    raise FileNotFoundError(f'Coloca {nombre} junto al cuaderno o en upload/.')
contenido = ruta.read_bytes()
origen = json.loads(contenido)
corte = date(2026, 11, 17)
print('Archivo:', ruta.name, '| Bytes:', len(contenido))
print('Huella SHA-256:', sha256(contenido).hexdigest())
for nombre_tabla, valor in origen.items():
    print(nombre_tabla, type(valor).__name__, len(valor) if isinstance(valor, list) else 'escalar')
tablas = {k: v for k, v in origen.items() if isinstance(v, list)}
tablas['proyectos'] = [dict(p, uid_cliente=c['uid']) for c in origen['clientes'] for p in c['proyectos']]
tablas['participaciones'] = [dict(p, uid_colaborador=a['uid_colaborador'])
                            for a in origen['asignaciones'] for p in a['proyectos']]
tablas['entregas'] = [dict(e, uid_activo=a['uid']) for a in origen['activos'] for e in a['asignaciones']]
tablas['evaluaciones'] = [
    dict(uid_colaborador=r['uid_colaborador'], codigo_habilidad=k, nivel=v)
    for r in origen['matriz'] for k,v in r['habilidades'].items()
]

Archivo: otc_backup_2026-11-17.json | Bytes: 221468
Huella SHA-256: a29ef7ac46a47ee5b7f3ff0e8e390bce82491da892b2016e088d3f8f849e7fa7
colaboradores list 107
usuarios list 6
habilidades list 71
roles list 55
departamentos list 6
matriz list 56
clientes list 8
asignaciones list 30
activos list 123
secuenciaOTC int escalar


## 2. Perfil de todas las columnas en Python
Un ausente es una clave inexistente, un nulo es un valor JSON `null` y un blanco es texto vacío después de quitar espacios. Un contenedor vacío se cuenta por separado. El conteo de distintos usa la representación JSON y conserva diferencias de tipo. El nivel de la matriz se observa en las evaluaciones registradas y cada atributo de los objetos anidados se perfila en su propia tabla.

In [2]:
def blanco(v):
    return isinstance(v,str) and not v.strip()

def serializar(v):
    return json.dumps(v, sort_keys=True, ensure_ascii=False)

perfil = {}
for tabla, filas in tablas.items():
    campos = sorted(set().union(*(r.keys() for r in filas)))
    print(f'\n{tabla} | {len(filas)} filas y {len(campos)} campos')
    print('campo | tipos observados | ausentes | nulos | blancos | cont. vacíos | distintos')
    for campo in campos:
        valores = [fila[campo] for fila in filas if campo in fila]
        tipos = ', '.join(sorted({type(v).__name__ for v in valores}))
        medidas = (len(filas)-len(valores), sum(v is None for v in valores),
                   sum(blanco(v) for v in valores),
                   sum(isinstance(v,(dict,list)) and not v for v in valores),
                   len({serializar(v) for v in valores if v is not None and not blanco(v)})
                   if campo != 'pass' else 'omitido')
        perfil[tabla,campo] = (tipos, *medidas)
        print(campo, tipos, *medidas, sep=' | ')
print('secuenciaOTC | entero | metadato de aplicación')


colaboradores | 107 filas y 47 campos
campo | tipos observados | ausentes | nulos | blancos | cont. vacíos | distintos
anexos | list | 0 | 0 | 0 | 107 | 1
aviso_infonavit | str | 0 | 0 | 0 | 0 | 3
aviso_manutencion | str | 0 | 0 | 2 | 0 | 3
ben_nombre | str | 0 | 0 | 3 | 0 | 5
ben_parentesco | str | 0 | 0 | 3 | 0 | 5
ben_porcentaje | str | 0 | 0 | 104 | 0 | 2
ciudad | str | 0 | 0 | 0 | 0 | 25
comision_nomina | float, int | 0 | 0 | 0 | 0 | 52
curp | str | 0 | 0 | 0 | 0 | 1
depto | str | 0 | 0 | 95 | 0 | 2
doc_acta | str | 0 | 0 | 0 | 0 | 2
doc_banco | str | 0 | 0 | 0 | 0 | 2
doc_compdom | str | 0 | 0 | 0 | 0 | 2
doc_contrato | str | 0 | 0 | 92 | 0 | 2
doc_csf | str | 0 | 0 | 0 | 0 | 2
doc_curp | str | 0 | 0 | 0 | 0 | 2
doc_foto | str | 0 | 0 | 0 | 0 | 2
doc_ine | str | 0 | 0 | 0 | 0 | 2
doc_nss | str | 0 | 0 | 0 | 0 | 2
doc_pasaporte | str | 0 | 0 | 16 | 0 | 2
doc_visa | str | 0 | 0 | 16 | 0 | 2
domicilio | str | 0 | 0 | 0 | 0 | 1
email | str | 0 | 0 | 0 | 0 | 1
estatus | str | 0 | 0 |

## 3. Validación funcional y bitácora automática
Se examinan dominios operativos, claves, fechas, uniones y consistencia aritmética. La fecha 1990-01-01 en participaciones se marca como candidata a centinela, pendiente de validación. Un fin blanco deja la vigencia indeterminada cuando faltan otras señales. Los conteos pueden compartir filas.

In [3]:
def sin_dato(v):
    return v is None or blanco(v)

def fecha(v):
    try:
        return date.fromisoformat(v)
    except (ValueError, TypeError):
        return None

bitacora = []
def registrar(tabla, campo, problema, casos, universo, prioridad, accion):
    bitacora.append((tabla,campo,problema,casos,universo,
                     f'{100*casos/universo:.1f}%' if universo else '—',prioridad,accion))

for t,k in [('colaboradores','uid'),('colaboradores','id'),('usuarios','uid'),
            ('habilidades','codigo'),('roles','codigo'),('departamentos','codigo'),
            ('clientes','uid'),('clientes','codigo'),('activos','codigo'),
            ('activos','uid'),('matriz','uid_colaborador'),('asignaciones','uid_colaborador')]:
    frecuencias=Counter(r.get(k) for r in tablas[t] if not sin_dato(r.get(k)))
    registrar(t,k,'Filas con clave repetida',sum(n for n in frecuencias.values() if n>1),
              len(tablas[t]),'Alta','Validar identidad y resolver con el responsable')

col=tablas['colaboradores']
personas={r['uid'] for r in col}
habilidades={r['codigo'] for r in tablas['habilidades']}
def normal(v):
    return ' '.join((v or '').split()).casefold()
proyectos=Counter((p['uid_cliente'],normal(p['nombre'])) for p in tablas['proyectos'])
def resolver(p):
    cliente, sep, nombre=p.get('proyecto_val','').partition('|')
    return proyectos[cliente,normal(nombre)] if sep else 0

for t,k,ref in [('participaciones','uid_colaborador',personas),
                ('entregas','uid_colab',personas),('evaluaciones','uid_colaborador',personas),
                ('evaluaciones','codigo_habilidad',habilidades)]:
    registrar(t,k,'Referencia sin catálogo',sum(r.get(k) not in ref for r in tablas[t]),
              len(tablas[t]),'Alta','Conservar en revisión y solicitar correspondencia')
for t, k in [('colaboradores','depto'),('colaboradores','doc_contrato'),
             ('roles','depto'),('departamentos','centroCostos'),('departamentos','responsable'),
             ('activos','fcompra'),('activos','fgarantia'),('activos','proveedor'),
             ('proyectos','inicio'),('proyectos','fin'),('participaciones','fin')]:
    registrar(t,k,'Campo sin dato',sum(sin_dato(r.get(k)) for r in tablas[t]),len(tablas[t]),
              'Media','Confirmar aplicabilidad y completar desde el sistema dueño')
for t, campos in [('colaboradores',['fechanac','fechainc','fechabaja']),
                  ('activos',['fcompra','fgarantia']),('entregas',['f_ini','f_fin']),
                  ('proyectos',['inicio','fin']),('participaciones',['inicio','fin'])]:
    for k in campos:
        registrar(t,k,'Fecha no ISO',sum(not sin_dato(r.get(k)) and fecha(r.get(k)) is None
                                        for r in tablas[t]),len(tablas[t]),
                  'Alta','Validar formato original y corregir con respaldo')
for t, ini, fin in [('proyectos','inicio','fin'),('participaciones','inicio','fin'),
                    ('entregas','f_ini','f_fin'),('colaboradores','fechainc','fechabaja')]:
    pares=[(fecha(r.get(ini)),fecha(r.get(fin))) for r in tablas[t]]
    registrar(t,f'{ini}/{fin}','Fin anterior al inicio',
              sum(i is not None and f is not None and f<i for i,f in pares),len(pares),
              'Alta','Validar las fechas con el responsable')
bajas=[r for r in col if r['estatus']=='Baja']
registrar('colaboradores','fechabaja','Baja sin fecha',sum(sin_dato(r.get('fechabaja')) for r in bajas),
          len(bajas),'Alta','Recuperar fecha de baja con RH')
participaciones=tablas['participaciones']
registrar('participaciones','inicio','Fecha candidata a centinela',
          sum(p.get('inicio')=='1990-01-01' for p in participaciones),len(participaciones),
          'Alta','Confirmar con operaciones y conservar el valor original')
for n, etiqueta in [(0,'Proyecto no encontrado'),(2,'Proyecto ambiguo')]:
    registrar('participaciones','proyecto_val',etiqueta,
              sum(resolver(p)==0 if n==0 else resolver(p)>1 for p in participaciones),
              len(participaciones),'Alta','Resolver con identificador de proyecto estable')
for total, partes in [
    ('sueldo_neto_total',('sueldo_imss','sueldo_asimilados','sueldo_complementos')),
    ('sueldo_total_integrado',('sueldo_neto_total','comision_nomina','impuestos_imss'))
]:
    registrar('colaboradores',total,'Diferencia en el cuadre mayor a 0.01',
              sum(abs(r[total]-sum(r[k] for k in partes))>0.01000001 for r in col),
              len(col),'Alta','Conciliar con nómina antes de usar para costeo')
registrar('activos','costo','Costo cero',sum(r['costo']==0 for r in tablas['activos']),
          len(tablas['activos']),'Media','Confirmar origen del cero y separar costos desconocidos')
nombres_roles={normal(r['nombre']) for r in tablas['roles']}
registrar('colaboradores','puesto','Nombre de puesto sin coincidencia normalizada',
          sum(normal(r['puesto']) not in nombres_roles for r in col),len(col),'Media',
          'Validar equivalencias con RH y asignar código del rol')
for t,k in [('colaboradores','estatus'),('habilidades','tipo'),('roles','nivel'),
            ('activos','tipo'),('proyectos','estatus'),('participaciones','tipo'),
            ('participaciones','estatus'),('evaluaciones','nivel')]:
    print(t,k,dict(Counter(r.get(k) for r in tablas[t])))
print('tabla | campo | hallazgo | casos | universo | tasa | prioridad | acción')
for r in bitacora:
    print(*r, sep=' | ')

colaboradores estatus {'Activo': 75, 'Baja': 32}
habilidades tipo {'HARD': 67, 'SOFT': 4}
roles nivel {'Director': 3, 'Analista': 14, 'Junior': 15, 'Senior': 16, 'Gerente': 5, 'Practicante': 1, 'Jefe de Área': 1}
activos tipo {'Laptop': 122, 'Monitor': 1}
proyectos estatus {'En Curso': 7, 'Por Iniciar': 5}
participaciones tipo {'Asignación': 15, 'Proyecto': 16}
participaciones estatus {'En Curso': 31}
evaluaciones nivel {'Avanzado': 234, 'Experto': 249, 'Básico': 687, 'Intermedio': 369}
tabla | campo | hallazgo | casos | universo | tasa | prioridad | acción
colaboradores | uid | Filas con clave repetida | 0 | 107 | 0.0% | Alta | Validar identidad y resolver con el responsable
colaboradores | id | Filas con clave repetida | 0 | 107 | 0.0% | Alta | Validar identidad y resolver con el responsable
usuarios | uid | Filas con clave repetida | 0 | 6 | 0.0% | Alta | Validar identidad y resolver con el responsable
habilidades | codigo | Filas con clave repetida | 0 | 71 | 0.0% | Alta | Validar 

## 4. Exploración relacional en SQL
SQLite funciona en memoria. La proyección relacional incluye las claves para probar relaciones y excluye cuentas, nombres de personas, contactos y salarios. Las consultas verifican cobertura, cardinalidades y vínculos. La cobertura histórica representa presencia de registros en el respaldo.

In [4]:
db=sqlite3.connect(':memory:')
db.execute('PRAGMA foreign_keys=ON')
seleccion = {
    'colaboradores': ('uid','estatus'),
    'clientes': ('uid','estatus'),
    'proyectos': ('uid_cliente','nombre','estatus','inicio','fin'),
    'participaciones': ('uid_colaborador','proyecto_val','estatus','inicio','fin'),
    'activos': ('uid','tipo','fcompra','fgarantia'),
    'entregas': ('uid_activo','uid_colab','f_ini','f_fin'),
    'habilidades': ('codigo','tipo'),
    'evaluaciones': ('uid_colaborador','codigo_habilidad','nivel')
}
for t, campos in seleccion.items():
    definicion=', '.join(f'{k} TEXT' for k in campos)
    db.execute(f'CREATE TABLE {t} ({definicion})')
    marcas=', '.join('?' for _ in campos)
    db.executemany(f'INSERT INTO {t} VALUES ({marcas})',
                   (tuple(r.get(k) for k in campos) for r in tablas[t]))
db.create_function('normalizar',1,normal)
consultas = {
    'Cobertura de registros por estatus': '''
        SELECT c.estatus, COUNT(*) AS personas,
               SUM(EXISTS(SELECT 1 FROM participaciones p WHERE p.uid_colaborador=c.uid)) AS con_participacion,
               SUM(EXISTS(SELECT 1 FROM evaluaciones e WHERE e.uid_colaborador=c.uid)) AS con_evaluacion
        FROM colaboradores c GROUP BY c.estatus ORDER BY c.estatus
    ''',
    'Referencias a proyectos por número de coincidencias': '''
        WITH cruces AS (
            SELECT a.rowid AS fila, COUNT(p.rowid) AS coincidencias
            FROM participaciones a
            LEFT JOIN proyectos p
              ON p.uid_cliente=substr(a.proyecto_val,1,instr(a.proyecto_val,'|')-1)
             AND normalizar(p.nombre)=normalizar(substr(a.proyecto_val,instr(a.proyecto_val,'|')+1))
            GROUP BY a.rowid
        )
        SELECT coincidencias, COUNT(*) AS participaciones
        FROM cruces GROUP BY coincidencias ORDER BY coincidencias
    ''',
    'Evaluaciones por tipo y nivel': '''
        SELECT COALESCE(h.tipo,'Sin catálogo') AS tipo,e.nivel,COUNT(*) AS evaluaciones
        FROM evaluaciones e LEFT JOIN habilidades h ON e.codigo_habilidad=h.codigo
        GROUP BY h.tipo,e.nivel ORDER BY tipo,e.nivel
    ''',
    'Cobertura de entregas por tipo de equipo': '''
        SELECT a.tipo, COUNT(*) AS equipos,
               SUM(EXISTS(SELECT 1 FROM entregas e WHERE e.uid_activo=a.uid)) AS con_entrega
        FROM activos a GROUP BY a.tipo ORDER BY a.tipo
    '''
}
for titulo, sql in consultas.items():
    print(titulo)
    resultado=db.execute(sql)
    print(*(c[0] for c in resultado.description),sep=' | ')
    for fila in resultado:
        print(*fila,sep=' | ')
    print()
assert db.execute('SELECT COUNT(*) FROM participaciones').fetchone()[0] == len(participaciones)
assert db.execute('SELECT COUNT(*) FROM evaluaciones').fetchone()[0] == len(tablas['evaluaciones'])
db.close()

Cobertura de registros por estatus
estatus | personas | con_participacion | con_evaluacion
Activo | 75 | 30 | 47
Baja | 32 | 0 | 7

Referencias a proyectos por número de coincidencias
coincidencias | participaciones
1 | 31

Evaluaciones por tipo y nivel
tipo | nivel | evaluaciones
HARD | Avanzado | 229
HARD | Básico | 666
HARD | Experto | 233
HARD | Intermedio | 343
SOFT | Avanzado | 5
SOFT | Básico | 21
SOFT | Experto | 16
SOFT | Intermedio | 26

Cobertura de entregas por tipo de equipo
tipo | equipos | con_entrega
Laptop | 122 | 90
Monitor | 1 | 1



## 5. Interpretación de la calidad
Las 30 estructuras de asignación se convierten en 31 participaciones. La repetición de una persona en el detalle puede reflejar varias participaciones y se debe verificar con fechas y dedicación. Para estimar disponibilidad se necesitan fechas confiables, capacidad y trabajo interno. Una evaluación ausente no equivale a nivel cero. Un costo integrado de nómina permite diseñar una estimación de costo cuando se confirme la unidad y se documenten las horas. Las preguntas de negocio y sus fórmulas propuestas están en el informe y quedan pendientes de respuesta.